# Phase 2: Data Preprocessing & Feature Engineering

**Project**: Personalized Movie Recommendation System  
**Repository**: `Dp8453/Personalized-movie-recommendation-system`  
**Output**: `data/processed/clean_movies.csv` (git-ignored)  

---  
### Objectives of Phase 2:
1. Parse JSON-like string metadata (`genres`, `keywords`, `cast`, `crew`) using `ast.literal_eval` safely.
2. Extract high-signal feature tokens: genres, keywords, top 3 lead actors, and the Director.
3. Perform entity tokenization (e.g. collapse spaces in multi-word names like `"Johnny Depp"` -> `"JohnnyDepp"`).
4. Clean plot synopses and impute missing `overview` values with empty strings.
5. Construct unified, normalized `tags` text representation for each movie.
6. Execute 14 rigorous data quality checks and export the clean dataset.

## 1. Import Required Libraries & Modules

In [ ]:
import os
import sys
import pandas as pd
import ast

# Ensure src module can be imported from parent directory
sys.path.append(os.path.abspath('..'))
from src.data_loader import load_movies
from src.preprocessor import (
    safe_eval_json,
    extract_genres,
    extract_keywords,
    extract_top_cast,
    extract_director,
    collapse_spaces,
    clean_overview,
    build_tags,
    preprocess_data,
)

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)
print('Imports successful.')

## 2. Load Phase 1 Merged Dataset

In [ ]:
raw_df = load_movies()
print(f'Loaded raw dataset shape: {raw_df.shape}')

## 3. Raw Metadata Inspection

In [ ]:
sample_movie = raw_df.iloc[0]
print(f"Title: {sample_movie['title']}\n")
print(f"Raw Genres: {sample_movie['genres']}\n")
print(f"Raw Keywords (Truncated): {str(sample_movie['keywords'])[:150]}...\n")
print(f"Raw Cast (Truncated): {str(sample_movie['cast'])[:150]}...\n")
print(f"Raw Crew (Truncated): {str(sample_movie['crew'])[:150]}...")

## 4. Demonstrate Feature Extraction & Tokenization

In [ ]:
# Demonstrate extraction functions on sample movie
genres = extract_genres(sample_movie['genres'])
keywords = extract_keywords(sample_movie['keywords'])
cast = extract_top_cast(sample_movie['cast'], top_n=3)
director = extract_director(sample_movie['crew'])
overview_tokens = clean_overview(sample_movie['overview'])

print(f"Extracted Genres: {genres}")
print(f"Extracted Keywords (First 5): {keywords[:5]}")
print(f"Extracted Top 3 Cast: {cast}")
print(f"Collapsed Cast Tokens: {collapse_spaces(cast)}")
print(f"Extracted Director: {director}")
print(f"Collapsed Director Token: {collapse_spaces(director)}")

## 5. Execute Complete Preprocessing Pipeline

In [ ]:
clean_df = preprocess_data(raw_df)
print(f'Processed dataset shape: {clean_df.shape}')

## 6. Before / After Comparison Examples

In [ ]:
print('=== BEFORE PREPROCESSING (Raw Metadata) ===')
print(raw_df[['id', 'title', 'overview', 'genres', 'cast']].head(2))

print('\n=== AFTER PREPROCESSING (Clean Features & Unified Tags) ===')
print(clean_df[['id', 'title', 'genres', 'cast', 'director', 'tags']].head(2))

### 6.1 Sample Unified `tags` Output

In [ ]:
for i in range(2):
    print(f"--- Movie {i+1}: {clean_df['title'].iloc[i]} ---")
    print(f"TAGS:\n{clean_df['tags'].iloc[i]}\n")

## 7. Data Quality Checks (14 Assertions)

In [ ]:
print('=== RUNNING DATA QUALITY VERIFICATION ===')
print(f"1. Rows before preprocessing : {len(raw_df)}")
print(f"2. Rows after preprocessing  : {len(clean_df)}")
print(f"3. Unexpected movie loss    : {len(raw_df) - len(clean_df)}")
print(f"4. Duplicate IDs in clean df: {clean_df['id'].duplicated().sum()}")
print(f"5. Raw missing overviews    : {raw_df['overview'].isnull().sum()}")
print(f"6. Processed null overviews : {clean_df['overview'].isnull().sum()}")
print(f"7. Empty tags count         : {(clean_df['tags'].str.strip() == '').sum()}")
print(f"8. Tags data type check     : {(clean_df['tags'].apply(type) == str).all()}")
print(f"9. 'nan' token in tags count: {clean_df['tags'].apply(lambda x: 'nan' in str(x).split()).sum()}")
print(f"10. 'none' token in tags cnt : {clean_df['tags'].apply(lambda x: 'none' in str(x).split()).sum()}")

cast_counts = clean_df['cast'].apply(lambda x: len(x) if isinstance(x, list) else len(ast.literal_eval(str(x))))
print(f"11. Max cast members/movie  : {cast_counts.max()}")

dir_counts = clean_df['director'].apply(lambda x: len(x) if isinstance(x, list) else len(ast.literal_eval(str(x))))
print(f"12. Max directors/movie     : {dir_counts.max()}")
print(f"13. Movies with director    : {(dir_counts > 0).sum()}")

genre_counts = clean_df['genres'].apply(lambda x: len(x) if isinstance(x, list) else len(ast.literal_eval(str(x))))
print(f"14. Movies with genres      : {(genre_counts > 0).sum()}")